# 하이퍼파리미터 튜닝

In [ ]:
# ----------------------------------------------------------------------
# 하이퍼파라미터 튜닝 (Hyperparameter Tuning) 정의
# ----------------------------------------------------------------------

# 1. 정의:
# - 모델 학습 과정에서 데이터로부터 자동으로 학습되지 않고, 사용자가 직접 설정해야 하는 값(파라미터)을 최적으로 찾는 과정.
# - 예시: K-NN의 K값, 랜덤 포레스트의 트리 개수(n_estimators), 학습률(learning_rate), 규제 강도(alpha 등).

# 2. 목적:
# - 과적합/과소적합 방지 및 모델의 일반화 성능(Generalization Performance) 극대화.
# - 최적의 하이퍼파라미터 조합을 찾아 가장 높은 검증 점수(Validation Score)를 달성하는 것.

# ----------------------------------------------------------------------
# 주요 튜닝 기법
# ----------------------------------------------------------------------
# 1. Grid Search (그리드 탐색)
# - 원리: 사용자가 지정한 하이퍼파라미터들의 모든 조합을 격자(Grid) 형태로 탐색하여 가장 좋은 성능을 내는 조합을 찾는 방식.
# - 특징:
#   - 장점: 모든 조합을 시도하므로, 최적의 조합을 찾을 확률이 높음.
#   - 단점: 탐색 공간이 넓거나 하이퍼파라미터 개수가 많아지면 기하급수적으로 시간이 증가 (비효율적).

# 2. Random Search (랜덤 탐색, 참고)
# - 원리: 지정된 탐색 공간 내에서 하이퍼파라미터 조합을 무작위로 샘플링하여 성능을 비교.
# - 특징: Grid Search보다 적은 시간으로 최적에 가까운 조합을 찾을 가능성이 높아, Grid Search의 대안으로 많이 사용됨.

# 3. Optuna (베이지안 최적화 기반)
# - 원리: 이전 시도에서 얻은 정보를 바탕으로 다음 시도에 가장 좋은 성능을 낼 가능성이 높은 하이퍼파라미터 조합을 '지능적으로' 선택하여 탐색. (베이지안 최적화 원리 활용)
# - 특징:
#   - 장점: Grid/Random Search보다 훨씬 적은 반복만으로 최적의 조합을 찾을 수 있어 탐색 시간이 단축됨 (가장 효율적).
#   - 특징: 'Define-by-Run' 방식을 사용하여 탐색 공간을 동적으로 정의할 수 있음.

## Grid Search(그리드 탐색)

In [ ]:
# ----------------------------------------------------------------------
# Grid Search (그리드 탐색) 정의 및 원리
# ----------------------------------------------------------------------

# 1. 정의:
# - 사용자가 지정한 '하이퍼파라미터'들의 모든 가능한 조합(격자, Grid)을 '완전 탐색'하여 최적의 성능을 내는 조합을 찾는 방식.
# - 지도 학습(Supervised Learning) 환경에서 모델의 일반화 성능을 높이는 데 주로 사용됨.

# 2. 핵심 원리:
# - 탐색 공간 정의: 튜닝할 각 하이퍼파라미터에 대해 후보 값들의 리스트를 지정.
# - 완전 탐색: 지정된 리스트의 '데카르트 곱(Cartesian Product)'에 해당하는 모든 조합을 하나씩 시도.
# - 교차 검증 (CV): 각 조합에 대해 K-Fold 교차 검증을 수행하여 일반화 성능(평균 점수)을 평가.
# - 최적 선택: 가장 높은 평균 교차 검증 점수를 기록한 하이퍼파라미터 조합을 최종 선택.

# 3. 장점:
# - 간결성: 구현이 매우 간단하고 직관적.
# - 확실성: 정의된 탐색 공간 내에서는 최적의 조합을 놓치지 않고 찾음.

# 4. 단점:
# - 비효율성: 탐색 공간(하이퍼파라미터 개수나 후보 값의 범위)이 넓어지면, 탐색 시간이 기하급수적으로 증가함.
# - 자원 소모: 계산 비용과 시간이 많이 소모되므로, 대규모 데이터셋이나 복잡한 모델에는 부적합할 수 있음.
# - Random Search나 Optuna 같은 지능형 탐색 기법에 비해 시간이 비효율적임.

# ----------------------------------------------------------------------
# Python에서의 구현 (Scikit-learn)
# ----------------------------------------------------------------------

# - 클래스: sklearn.model_selection.GridSearchCV
# - 주요 매개변수:
#   - estimator: 튜닝할 머신러닝 모델 객체.
#   - param_grid: 하이퍼파라미터 후보 값들을 담은 딕셔너리.
#   - cv: 사용할 교차 검증 폴드(Fold) 개수 (예: cv=5).
#   - scoring: 평가 지표 (예: 'accuracy', 'f1', 'neg_mean_squared_error').

# - 주요 결과 속성:
#   - .best_params_: 최적의 성능을 낸 하이퍼파라미터 조합 딕셔너리.
#   - .best_score_: 최적의 조합으로 얻은 최고 교차 검증 평균 점수.
#   - .best_estimator_: 최적의 하이퍼파라미터로 재학습된 최종 모델 객체.

In [1]:
# --------------------
# 라이브러리 및 데이터 준비
# --------------------
import numpy as np
from sklearn.model_selection import GridSearchCV # Grid Search 함수
from sklearn.ensemble import RandomForestClassifier # 탐색 대상 모델
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Iris 데이터셋 로드
iris = load_iris()
X, y = iris.data, iris.target

# 데이터 분할 (Grid Search 내부에서 교차 검증을 수행하므로 분할이 필수는 아님)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("--- 데이터 준비 완료 ---")

--- 데이터 준비 완료 ---


In [2]:
# --------------------
#  2단계: 탐색 공간 (Parameter Grid) 정의
# --------------------
# n_estimators: 트리의 개수 (100, 200)
# max_depth: 트리의 최대 깊이 (3, 5, 7)
# 총 시도 횟수 = 2 * 3 = 6가지 조합
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'min_samples_leaf': [10, 20] # leaf 노드가 되기 위한 최소 샘플 수
}

print("\n--- 2단계: 탐색할 하이퍼파라미터 조합 수 ---")
# n_estimators(2) * max_depth(3) * min_samples_leaf(2) = 12가지 조합
total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"총 시도할 조합의 수: {total_combinations}가지")


--- 2단계: 탐색할 하이퍼파라미터 조합 수 ---
총 시도할 조합의 수: 12가지


In [3]:
# --------------------
# 3단계: Grid Search 실행 및 최적 조합 찾기
# --------------------
# estimator: 사용할 모델 (랜덤 포레스트)
# param_grid: 정의된 탐색 공간
# cv=5: 5-Fold 교차 검증 사용
# scoring='accuracy': 평가 지표는 정확도(Accuracy) 사용
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42), 
    param_grid=param_grid, 
    cv=5, 
    scoring='accuracy',
    n_jobs=-1 # 모든 코어 사용
)

# 학습 데이터에 Grid Search 실행
grid_search.fit(X_train, y_train)

# --------------------
# 4단계: 결과 확인
# --------------------
print("\n--- 4단계: 최적의 하이퍼파라미터 조합 및 성능 ---")
# 최적의 하이퍼파라미터 조합 (Best Parameters)
print("Best Parameters:", grid_search.best_params_)

# 해당 조합으로 얻은 최고 교차 검증 점수 (Best Score)
print(f"Best CV Score (Accuracy): {grid_search.best_score_:.4f}")

# 최종 모델: 최적 하이퍼파라미터로 재학습된 모델
best_model = grid_search.best_estimator_
final_test_score = best_model.score(X_test, y_test)
print(f"Final Test Score: {final_test_score:.4f}")


--- 4단계: 최적의 하이퍼파라미터 조합 및 성능 ---
Best Parameters: {'max_depth': 3, 'min_samples_leaf': 10, 'n_estimators': 100}
Best CV Score (Accuracy): 0.9333
Final Test Score: 1.0000


## Random Search (랜덤 탐색)

In [ ]:
# ----------------------------------------------------------------------
#  Random Search (랜덤 탐색) 정의 및 원리
# ----------------------------------------------------------------------

# 1. 정의:
# - 사용자가 정의한 하이퍼파라미터의 탐색 공간(범위 또는 분포) 내에서 '정해진 횟수'만큼 '무작위'로 조합을 샘플링하여 성능을 평가하는 방식.
# - Grid Search의 대안으로, 탐색 시간 대비 효율을 높이는 데 목적이 있음.

# 2. 핵심 원리:
# - 탐색 공간 정의: 각 하이퍼파라미터의 후보 값 리스트나 확률 분포(예: 균일 분포, 정규 분포)를 정의.
# - 무작위 샘플링: 사용자가 지정한 횟수(n_iter)만큼 조합을 무작위로 추출하여 시도.
# - 교차 검증 (CV): 추출된 각 조합에 대해 K-Fold 교차 검증을 수행하여 일반화 성능(평균 점수)을 평가.
# - 최적 선택: 시도 횟수 내에서 가장 높은 평균 점수를 기록한 조합을 선택.

# 3. Grid Search 대비 장점:
# - 효율성: 실제로 성능에 큰 영향을 미치는 파라미터 값을 탐색할 확률이 높음. (Grid Search는 중요하지 않은 파라미터 축을 불필요하게 탐색함).
# - 속도: 탐색 횟수(n_iter)를 직접 제어하므로, Grid Search보다 훨씬 빠르게 튜닝을 완료할 수 있음.

# 4. 단점:
# - 불확실성: Grid Search와 달리 정의된 탐색 공간 전체를 확인하지 않기 때문에, 운이 나쁘면 최적의 조합을 놓칠 수 있음.
# - 탐색 횟수 의존성: n_iter 설정이 너무 작으면 성능이 떨어질 수 있음.

# ----------------------------------------------------------------------
#  Python에서의 구현 (Scikit-learn)
# ----------------------------------------------------------------------

# - 클래스: sklearn.model_selection.RandomizedSearchCV
# - 주요 매개변수:
#   - param_distributions: 하이퍼파라미터의 확률 분포 또는 후보 리스트를 담은 딕셔너리.
#   - n_iter: 무작위로 샘플링할 횟수 (탐색 시간 제어).
#   - cv: 사용할 교차 검증 폴드 개수.
#   - scoring: 평가 지표.

In [4]:
# --------------------
#  라이브러리 및 데이터 준비
# --------------------
import numpy as np
from sklearn.model_selection import RandomizedSearchCV # Random Search 함수
from sklearn.ensemble import RandomForestClassifier # 탐색 대상 모델
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from scipy.stats import randint, uniform # 랜덤 샘플링을 위한 분포 함수

# Iris 데이터셋 로드
iris = load_iris()
X, y = iris.data, iris.target

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("--- 데이터 준비 완료 ---")

--- 데이터 준비 완료 ---


In [5]:
# --------------------
#  2단계: 탐색 공간 (Parameter Distribution) 정의
# --------------------
# n_estimators: 100부터 500까지의 정수 범위에서 균일하게 샘플링
# max_depth: 3부터 10까지의 정수 범위에서 균일하게 샘플링
# min_samples_split: 2부터 10까지의 정수 범위에서 균일하게 샘플링
param_distributions = {
    'n_estimators': randint(low=100, high=500),
    'max_depth': randint(low=3, high=10),
    'min_samples_split': randint(low=2, high=10) 
}

print("\n--- 2단계: 탐색할 하이퍼파라미터 분포 정의 완료 ---")


--- 2단계: 탐색할 하이퍼파라미터 분포 정의 완료 ---


In [6]:
# --------------------
#  3단계: Random Search 실행 및 최적 조합 찾기
# --------------------
# estimator: 사용할 모델 (랜덤 포레스트)
# param_distributions: 정의된 탐색 공간 (확률 분포)
# n_iter=20: 무작위로 20가지 조합만 샘플링하여 시도
# cv=5: 5-Fold 교차 검증 사용
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42), 
    param_distributions=param_distributions, 
    n_iter=20, # Grid Search와 달리 시도 횟수를 제한
    cv=5, 
    scoring='accuracy',
    random_state=42,
    n_jobs=-1 
)

# 학습 데이터에 Random Search 실행
random_search.fit(X_train, y_train)

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....x7ec65524d010>, 'min_samples_split': <scipy.stats....x7ec65524db50>, 'n_estimators': <scipy.stats....x7ec65524db10>}"
,n_iter,20
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [7]:
# --------------------
# 4 단계: 결과 확인
# --------------------
print("\n--- 4단계: 최적의 하이퍼파라미터 조합 및 성능 ---")
# 최적의 하이퍼파라미터 조합 (Best Parameters)
print("Best Parameters:", random_search.best_params_)

# 해당 조합으로 얻은 최고 교차 검증 점수 (Best Score)
print(f"Best CV Score (Accuracy): {random_search.best_score_:.4f}")

# 최종 모델
best_model = random_search.best_estimator_
final_test_score = best_model.score(X_test, y_test)
print(f"Final Test Score: {final_test_score:.4f}")


--- 4단계: 최적의 하이퍼파라미터 조합 및 성능 ---
Best Parameters: {'max_depth': 9, 'min_samples_split': 5, 'n_estimators': 448}
Best CV Score (Accuracy): 0.9429
Final Test Score: 1.0000


## Optuna (베이지안 최적화 기반)

In [ ]:
# ----------------------------------------------------------------------
# Optuna (베이지안 최적화 기반) 정의 및 원리
# ----------------------------------------------------------------------

# 1. 정의:
# - 이전 시도의 평가 결과를 바탕으로 다음 시도에 가장 성능이 좋을 것으로 '예측'되는 조합을 '지능적으로' 선택하여 탐색하는 베이지안 최적화 기반 프레임워크.
# - Random Search보다 훨씬 적은 반복만으로 최적의 하이퍼파라미터를 찾을 수 있어 가장 효율적임.

# 2. 핵심 원리:
# - Define-by-Run: 탐색 공간을 미리 고정하지 않고, 튜닝 과정(Run-time) 중에 동적으로(조건에 따라) 정의할 수 있음.
# - 베이지안 최적화: 확률 모델(대리 모델)을 사용하여 탐색할 다음 위치를 결정.
#   - 탐험 (Exploration): 아직 시도해보지 않은 새로운 영역을 탐색.
#   - 활용 (Exploitation): 이전에 성능이 좋았던 영역 주변을 집중적으로 탐색.
#   - Optuna는 이 두 가지를 균형 있게 수행하여 효율을 극대화함.

# 3. 주요 요소:
# - Study (스터디): 전체 최적화 세션을 관리하는 객체. 최적화 방향(maximize/minimize)을 정의함.
# - Trial (시행): 한 번의 하이퍼파라미터 조합을 시도하고 평가하는 과정.
# - Objective Function (목적 함수): Optuna로부터 제안받은 조합으로 모델을 학습시키고, 최적화할 점수(Scalar Value)를 반환하는 함수. (Optuna가 이 값을 최대화 또는 최소화하려고 시도함)

# 4. 장점:
# - 높은 효율성: 비효율적인 영역 탐색을 피하여 시간과 자원을 크게 절약.
# - 유연성: 조건부 파라미터나 복잡한 탐색 공간을 동적으로 처리 가능.
# - 강력한 시각화: 탐색 과정을 시각적으로 분석할 수 있는 다양한 도구 제공.

In [8]:
# --------------------
#  라이브러리 및 데이터 준비
# --------------------
# pip install optuna scikit-learn
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score, train_test_split

# Iris 데이터셋 로드
iris = load_iris()
X, y = iris.data, iris.target

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("--- 데이터 준비 완료 ---")

--- 데이터 준비 완료 ---


In [9]:
# --------------------
# 2단계: 목적 함수 (Objective Function) 정의
# --------------------
def objective(trial):
    """
    Optuna로부터 하이퍼파라미터를 제안받아 모델을 평가하고 점수를 반환하는 함수.
    """
    
    # 1. 하이퍼파라미터 탐색 공간 정의 (trial.suggest_*)
    # Optuna는 이전에 평가된 결과를 바탕으로 다음에 시도할 최적의 값을 지능적으로 제안합니다.
    n_estimators = trial.suggest_int('n_estimators', 100, 500)  # 정수형 범위 탐색
    max_depth = trial.suggest_int('max_depth', 3, 10)           # 정수형 범위 탐색
    min_samples_split = trial.suggest_float('min_samples_split', 0.01, 0.1) # 실수형 범위 탐색
    
    # 2. 모델 생성 및 교차 검증 평가
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )
    
    # 5-Fold 교차 검증을 사용하여 정확도(Accuracy)를 평가
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    
    # 교차 검증 점수의 평균을 반환 (Optuna는 이 값을 최대화하려고 시도함)
    return score.mean()

print("\n--- 2단계: 목적 함수 정의 완료 ---")


--- 2단계: 목적 함수 정의 완료 ---


In [10]:
# --------------------
# 3단계: Optuna Study 실행 및 최적화
# --------------------
# direction="maximize": 목적 함수의 반환값(정확도)을 최대화하도록 설정
study = optuna.create_study(direction="maximize") 

# optimize 실행: 목적 함수를 호출하여 50번의 반복(trial) 동안 최적의 조합을 탐색
# n_trials=50: 총 50가지 조합만 시도 (Grid Search 대비 매우 효율적)
study.optimize(objective, n_trials=50, show_progress_bar=True) 

[I 2025-10-17 01:51:38,749] A new study created in memory with name: no-name-bface527-e31a-4a99-bd2b-38dbbeff103b


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-10-17 01:51:39,559] Trial 0 finished with value: 0.9428571428571428 and parameters: {'n_estimators': 182, 'max_depth': 10, 'min_samples_split': 0.05420355099842428}. Best is trial 0 with value: 0.9428571428571428.
[I 2025-10-17 01:51:41,365] Trial 1 finished with value: 0.9428571428571428 and parameters: {'n_estimators': 417, 'max_depth': 9, 'min_samples_split': 0.042130987848062636}. Best is trial 0 with value: 0.9428571428571428.
[I 2025-10-17 01:51:42,683] Trial 2 finished with value: 0.9428571428571428 and parameters: {'n_estimators': 317, 'max_depth': 9, 'min_samples_split': 0.013579541106331259}. Best is trial 0 with value: 0.9428571428571428.
[I 2025-10-17 01:51:44,132] Trial 3 finished with value: 0.9428571428571428 and parameters: {'n_estimators': 345, 'max_depth': 7, 'min_samples_split': 0.035731342921121435}. Best is trial 0 with value: 0.9428571428571428.
[I 2025-10-17 01:51:45,314] Trial 4 finished with value: 0.9428571428571428 and parameters: {'n_estimators': 285

In [11]:
# --------------------
# 4단계: 결과 확인
# --------------------
print("\n--- 4단계: 최적화 결과 확인 ---")
# 최적의 하이퍼파라미터 조합 (Best Parameters)
print("Best Parameters:", study.best_params)

# 해당 조합으로 얻은 최고 점수 (Best Value)
print(f"Best CV Score (Accuracy): {study.best_value:.4f}")

# 최종 모델 학습 (최적 파라미터를 사용하여 최종 모델 재학습)
best_params = study.best_params
final_model = RandomForestClassifier(**best_params, random_state=42)
final_model.fit(X_train, y_train)

final_test_score = final_model.score(X_test, y_test)
print(f"Final Test Score: {final_test_score:.4f}")


--- 4단계: 최적화 결과 확인 ---
Best Parameters: {'n_estimators': 182, 'max_depth': 10, 'min_samples_split': 0.05420355099842428}
Best CV Score (Accuracy): 0.9429
Final Test Score: 1.0000
